# STRATA v2 Cross-Model Benchmark on Google Colab (T4 GPU)

Evaluates 20B–35B GGUF baselines across **3 controlled context arms** aligned with STRATA v2's expanded 19-task suite (`v2_expanded`):
1. **`pure` (Zero-Shot, No `.md`):** Tests parametric weights only without any domain/SDK markdown.
2. **`skill_md` (Exact Loaded Domain `SKILL.md` in Chat):** Tests whether passing structured domain/SDK `.md` docs directly in the system prompt closes the gap on blind SDKs (`cryopulse`, `aetheris`), domain adherence (`Pillow` vs `OpenCV`), and wrong-domain refusals.
3. **`skill_md_haystack` (`SKILL.md` + 15-Domain Catalog Noise):** Embeds the unseen SDK docs at the midpoint of all 15 standard domain trees (`1,049` distractor operations) to measure needle-in-a-haystack degradation.

Models evaluated:
- **Qwen3.8-27B** (`unsloth/Qwen3.8-27B-GGUF`, UD-IQ3_S, 12.0 GB)
- **GPT-OSS 20B** (`unsloth/gpt-oss-20b-GGUF`, Q4_K_M, 11.6 GB)
- **Qwen3-Coder-30B-A3B** (`unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF`, Q3_K_S, 13.3 GB)
- **Qwen3.5-35B-A3B MoE** (`unsloth/Qwen3.5-35B-A3B-GGUF`, UD-IQ3_XXS, 13.1 GB)
- **Qwen3.5-27B Dense** (`unsloth/Qwen3.5-27B-GGUF`, Q3_K_S, 12.3 GB)

In [ ]:
# Cell 1: Verify Colab T4 GPU & cleanly clone STRATA
BRANCH = "main"  # Change to "dev" if benchmarking pre-release updates
!nvidia-smi
%cd /content
!rm -rf /content/STRATA
!git clone --branch "$BRANCH" https://github.com/MatthewNader2/STRATA.git /content/STRATA
%cd /content/STRATA
!pip -q install -r benchmarks/requirements-colab.txt

In [ ]:
# Cell 2: Download prebuilt CUDA 12.8 llama-server + cudart with integrity check
import json, os, pathlib, subprocess, urllib.request

req = urllib.request.Request(
    'https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=8',
    headers={'User-Agent': 'Mozilla/5.0'},
)
releases = json.loads(urllib.request.urlopen(req, timeout=20).read())
llama_dir = pathlib.Path('/content/llama_bin')
subprocess.run(['rm', '-rf', str(llama_dir)], check=True)
llama_dir.mkdir(parents=True, exist_ok=True)

chosen_assets = []
for rel in releases:
    assets = [a for a in rel.get('assets', []) if 'bin-ubuntu-cuda-12.8-x64.tar.gz' in a['name']]
    if any(a['name'].startswith('llama-') for a in assets) and any(a['name'].startswith('cudart-') for a in assets):
        chosen_assets = assets
        print('Using llama.cpp release:', rel['tag_name'])
        break
assert chosen_assets, 'No CUDA 12.8 Ubuntu llama.cpp release assets found!'

for asset in chosen_assets:
    dst = pathlib.Path('/content') / asset['name']
    ok_tar = dst.exists() and subprocess.run(['tar', '-tzf', str(dst)], capture_output=True).returncode == 0
    if not ok_tar:
        print('Downloading', asset['name'], '...')
        subprocess.run(['curl', '-fL', '--retry', '3', asset['browser_download_url'], '-o', str(dst)], check=True)
    subprocess.run(['tar', '-xzf', str(dst), '-C', str(llama_dir)], check=True)

server_bin = str(next(llama_dir.rglob('llama-server')))
so_dirs = sorted({str(p.parent) for p in llama_dir.rglob('*.so*')})
os.environ['LLAMA_SERVER'] = server_bin
os.environ['LD_LIBRARY_PATH'] = ':'.join(so_dirs + ['/usr/local/cuda/lib64', os.environ.get('LD_LIBRARY_PATH', '')]).rstrip(':')
print('LLAMA_SERVER =', os.environ['LLAMA_SERVER'])
!"$LLAMA_SERVER" --version

In [ ]:
# Cell 3: Arm 1 — Pure Zero-Shot (NO .md docs) across v2_expanded (19 tasks)
# Keep --resume cleanly and omit --cleanup-cache until the final arm if running all 3 arms back-to-back.
!python -u benchmarks/colab_eval.py \
  --llama-server "$LLAMA_SERVER" \
  --suite v2_expanded \
  --context-mode pure \
  --models qwen3.8-27b gpt-oss-20b qwen3-coder-30b-a3b qwen3.5-35b-a3b qwen3.5-27b \
  --out-dir test_logs/colab_v2_pure \
  --resume

In [ ]:
# Cell 4: Arm 2 — Exact Loaded Domain SKILL.md in System Prompt across v2_expanded (19 tasks)
!python -u benchmarks/colab_eval.py \
  --llama-server "$LLAMA_SERVER" \
  --suite v2_expanded \
  --context-mode skill_md \
  --haystack-chars 18000 \
  --models qwen3.8-27b gpt-oss-20b qwen3-coder-30b-a3b qwen3.5-35b-a3b qwen3.5-27b \
  --out-dir test_logs/colab_v2_skill_md \
  --resume

In [ ]:
# Cell 5: Arm 3 — SKILL.md + 15-Domain Haystack Stress across v2_haystack (Blind SDK Needles)
!python -u benchmarks/colab_eval.py \
  --llama-server "$LLAMA_SERVER" \
  --suite v2_haystack \
  --context-mode skill_md_haystack \
  --haystack-chars 18000 \
  --models qwen3.8-27b gpt-oss-20b qwen3-coder-30b-a3b qwen3.5-35b-a3b qwen3.5-27b \
  --out-dir test_logs/colab_v2_haystack \
  --cleanup-cache \
  --resume

In [ ]:
# Cell 6 (Optional): Additional v2_full extra tasks (cryo/aetheris implicit & diag, user_real, custom_algo)
!python -u benchmarks/colab_eval.py \
  --llama-server "$LLAMA_SERVER" \
  --suite v2_full \
  --context-mode skill_md \
  --haystack-chars 18000 \
  --models qwen3.8-27b gpt-oss-20b qwen3-coder-30b-a3b qwen3.5-35b-a3b qwen3.5-27b \
  --out-dir test_logs/colab_v2_full \
  --cleanup-cache \
  --resume

In [ ]:
# Cell 7: Ingest optional AI Studio Gemma 4 31B dumps, aggregate charts/tables & download archive
import os

for dump_file, ctx_mode, out_json in [
    ('gemma4_v2_pure.txt', 'pure', 'test_logs/colab_v2_pure/results_gemma4_31b_pure.json'),
    ('gemma4_v2_skill_md.txt', 'skill_md', 'test_logs/colab_v2_skill_md/results_gemma4_31b_skill_md.json'),
    ('gemma4_v2_haystack.txt', 'skill_md_haystack', 'test_logs/colab_v2_haystack/results_gemma4_31b_haystack.json'),
]:
    if os.path.exists(dump_file):
        !python -u benchmarks/ingest_aistudio.py --input "$dump_file" --suite v2_expanded --context-mode "$ctx_mode" --out "$out_json"

!python -u benchmarks/visualize_results.py test_logs/colab_v2_pure test_logs/colab_v2_skill_md test_logs/colab_v2_haystack --out-dir test_logs/visualizations
!zip -r /content/strata_colab_results.zip test_logs/colab_v2_pure test_logs/colab_v2_skill_md test_logs/colab_v2_haystack test_logs/visualizations
try:
    from google.colab import files
    files.download('/content/strata_colab_results.zip')
except Exception:
    print('Download /content/strata_colab_results.zip manually from Colab file pane.')